In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (keep original integer timestamps as well)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['datetime_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) File-level summaries
def file_summary(df, name):
    total_rows = len(df)
    if total_rows == 0:
        min_ts = None
        max_ts = None
        min_dt = None
        max_dt = None
        distinct_cmdb = 0
    else:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
        min_dt = pd.to_datetime(min_ts, unit='s', utc=True)
        max_dt = pd.to_datetime(max_ts, unit='s', utc=True)
        distinct_cmdb = df['cmdb_id'].nunique()
    return pd.DataFrame([{
        'file': name,
        'total_rows': total_rows,
        'min_timestamp': min_ts,
        'min_datetime_utc': min_dt,
        'max_timestamp': max_ts,
        'max_datetime_utc': max_dt,
        'distinct_cmdb_id_count': distinct_cmdb
    }])

metric_file_summary = file_summary(metric_df, 'metric.csv')
trace_file_summary = file_summary(trace_df, 'trace.csv')
log_file_summary = file_summary(log_df, 'log.csv')
error_file_summary = file_summary(error_df, 'error_logs.csv')

# 2) metric.csv: distinct (cmdb_id, kpi_name) pairs, top 20 by count, and global P95 for each pair
if not metric_df.empty:
    metric_groups = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        count='count',
        p95_value=lambda x: x.quantile(0.95)
    ).reset_index()
    metric_top20_by_count = metric_groups.sort_values('count', ascending=False).head(20)
    # For the requested output, sort these top-20-by-count rows by p95_value descending
    metric_top20_count_and_p95 = metric_top20_by_count.sort_values('p95_value', ascending=False).reset_index(drop=True)
else:
    metric_top20_count_and_p95 = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'count', 'p95_value'])

# 3) trace.csv: distinct (cmdb_id, trace_name) pairs, top 20 by count, global P95
if not trace_df.empty:
    trace_groups = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
        count='count',
        p95_value=lambda x: x.quantile(0.95)
    ).reset_index()
    trace_top20_by_count = trace_groups.sort_values('count', ascending=False).head(20)
    trace_top20_count_and_p95 = trace_top20_by_count.sort_values('p95_value', ascending=False).reset_index(drop=True)
else:
    trace_top20_count_and_p95 = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'count', 'p95_value'])

# 4) log.csv: distinct (cmdb_id, log_name) pairs, top 20 by count, global P95
if not log_df.empty:
    log_groups = log_df.groupby(['cmdb_id', 'log_name'])['value'].agg(
        count='count',
        p95_value=lambda x: x.quantile(0.95)
    ).reset_index()
    log_top20_by_count = log_groups.sort_values('count', ascending=False).head(20)
    log_top20_count_and_p95 = log_top20_by_count.sort_values('p95_value', ascending=False).reset_index(drop=True)
else:
    log_top20_count_and_p95 = pd.DataFrame(columns=['cmdb_id', 'log_name', 'count', 'p95_value'])

# 5) error_logs.csv: distinct cmdb_id values and count of error rows per cmdb_id (top 20).
# Also return up to 5 example message texts (truncate to 200 chars) with timestamps and cmdb_id.
if not error_df.empty:
    error_counts = error_df.groupby('cmdb_id').size().reset_index(name='error_count')
    error_top20_counts = error_counts.sort_values('error_count', ascending=False).head(20).reset_index(drop=True)
    # Prepare example messages
    error_df['message_str'] = error_df['message'].astype(str)
    error_df['message_trunc'] = error_df['message_str'].str.slice(0, 200)
    error_examples = error_df.sort_values('timestamp', ascending=False)[['timestamp', 'datetime_utc', 'cmdb_id', 'message_trunc']].head(5).reset_index(drop=True)
else:
    error_top20_counts = pd.DataFrame(columns=['cmdb_id', 'error_count'])
    error_examples = pd.DataFrame(columns=['timestamp', 'datetime_utc', 'cmdb_id', 'message_trunc'])

# Keep outputs compact: ensure ordering and column selection
metric_file_summary, trace_file_summary, log_file_summary, error_file_summary, metric_top20_count_and_p95.head(20), trace_top20_count_and_p95.head(20), log_top20_count_and_p95.head(20), error_top20_counts.head(20), error_examples.head(5)

```
Out[1]:
```
summary = (
    "Summary of telemetry files (UTC):\n"
    "- Files: metric.csv (1875 rows; ts range 1705668180 -> 1705669620 => 2024-01-19 12:43:00 -> 2024-01-19 13:07:00; 12 distinct cmdb_id), "
    "trace.csv (3800 rows; same ts range; 8 distinct cmdb_id), log.csv (458 rows; same ts range; 10 distinct cmdb_id), "
    "error_logs.csv (0 rows).\n\n"
    "Key metric highlights (top global P95 values):\n"
    "- adservice memory (kpi 'mem') has the highest P95 (~1.08e+08), followed by cartservice mem (~4.47e+07) and checkoutservice mem (~1.21e+07).\n"
    "- adservice diskio P95 is high (~7.5e+04). Other P95s (CPU, workload, latency) are much smaller in magnitude.\n\n"
    "Trace highlights:\n"
    "- checkoutservice dominates trace metrics by both count and P95: several row_count traces show high P95s (e.g., 283.6, 141, 117), "
    "and a frontend->checkoutservice duration p95 ~3.97s (mean ~2.65s), indicating elevated request latency hitting checkoutservice.\n\n"
    "Log highlights:\n"
    "- High log.row_count P95s for frontend (~2716.4), currencyservice (~2436.4), and cartservice (~794.8). "
    "All log.error_count series show P95 = 0.0 in available data.\n\n"
    "Error logs:\n"
    "- No error log rows available.\n\n"
    "Overall quick interpretation / next steps:\n"
    "- The most prominent resource anomaly is very high memory P95 in adservice; investigate adservice memory usage and recent changes.\n"
    "- Checkoutservice shows elevated trace durations (frontend->checkoutservice p95 ~4s) and high trace row counts; investigate latency and downstream calls for checkoutservice.\n"
    "- Check high disk I/O for adservice and high log volumes in frontend/currencyservice for related load patterns.\n"
    "- No error log evidence present in error_logs.csv to explain failures; collect application logs or increase retention if needed."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows  min_timestamp          min_datetime_utc  max_timestamp          max_datetime_utc  distinct_cmdb_id_count
0  metric.csv        1875     1705668180 2024-01-19 12:43:00+00:00     1705669620 2024-01-19 13:07:00+00:00                      12,         file  total_rows  min_timestamp          min_datetime_utc  max_timestamp          max_datetime_utc  distinct_cmdb_id_count
0  trace.csv        3800     1705668180 2024-01-19 12:43:00+00:00     1705669620 2024-01-19 13:07:00+00:00                       8,       file  total_rows  min_timestamp          min_datetime_utc  max_timestamp          max_datetime_utc  distinct_cmdb_id_count
0  log.csv         458     1705668180 2024-01-19 12:43:00+00:00     1705669620 2024-01-19 13:07:00+00:00                      10,              file  total_rows min_timestamp min_datetime_utc max_timestamp max_datetime_utc  distinct_cmdb_id_count
0  error_logs.csv           0          None             None          None             None                       0,             cmdb_id    kpi_name  count     p95_value
0         adservice         mem     25  1.077783e+08
1       cartservice         mem     25  4.467686e+07
2   checkoutservice         mem     25  1.214697e+07
3         adservice      diskio     25  7.497617e+04
4       cartservice    workload     25  2.627446e+01
5         adservice    workload     25  1.552699e+01
6   checkoutservice      socket     25  9.000000e+00
7   checkoutservice  latency-90     25  4.569212e+00
8       cartservice      socket     25  4.000000e+00
9         adservice      socket     25  4.000000e+00
10  checkoutservice  latency-50     25  2.771273e+00
11      cartservice         cpu     25  2.643995e+00
12        adservice         cpu     25  1.223571e+00
13  checkoutservice    workload     25  1.120730e+00
14  checkoutservice         cpu     25  5.045904e-01
15      cartservice  latency-90     25  5.443580e-03
16        adservice  latency-90     25  4.629975e-03
17      cartservice  latency-50     25  3.224301e-03
18        adservice  latency-50     25  3.012238e-03
19      cartservice      diskio     25  0.000000e+00,             cmdb_id                                trace_name  count   p95_value
0   checkoutservice      trace.from_checkoutservice.row_count     25  283.600000
1   checkoutservice        trace.to_checkoutservice.row_count     25  283.600000
2   checkoutservice                 trace.from_root.row_count     25  141.000000
3   checkoutservice        trace.to_currencyservice.row_count     25  117.000000
4   checkoutservice      trace.from_frontendservice.row_count     25   31.800000
5   checkoutservice   trace.from_frontendservice.duration_p95     25    3.966878
6   checkoutservice  trace.from_frontendservice.duration_mean     25    2.646320
7   checkoutservice              trace.from_root.duration_p95     25    0.275308
8   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.210122
9   checkoutservice    trace.to_checkoutservice.duration_mean     25    0.210122
10  checkoutservice     trace.to_checkoutservice.duration_p95     25    0.206567
11  checkoutservice   trace.from_checkoutservice.duration_p95     25    0.206567
12  checkoutservice             trace.from_root.duration_mean     25    0.198345
13  checkoutservice    trace.to_currencyservice.duration_mean     25    0.001224
14  checkoutservice     trace.to_currencyservice.duration_p95     25    0.000373
15  checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000
16  checkoutservice                trace.from_root.error_rate     25    0.000000
17  checkoutservice     trace.from_frontendservice.error_rate     25    0.000000
18  checkoutservice       trace.to_checkoutservice.error_rate     25    0.000000
19  checkoutservice       trace.to_currencyservice.error_rate     25    0.000000,                   cmdb_id         log_name  count  p95_value
0                frontend    log.row_count     25     2716.4
1         currencyservice    log.row_count     25     2436.4
2             cartservice    log.row_count     25      794.8
3   recommendationservice    log.row_count     25      587.2
4         shippingservice    log.row_count     25      484.8
5               adservice    log.row_count     25      461.4
6         checkoutservice    log.row_count     25       96.4
7          paymentservice    log.row_count     25       65.6
8            emailservice    log.row_count     25       32.0
9                   redis    log.row_count      4        5.0
10            cartservice  log.error_count     25        0.0
11              adservice  log.error_count     25        0.0
12               frontend  log.error_count     25        0.0
13           emailservice  log.error_count     25        0.0
14        checkoutservice  log.error_count     25        0.0
15        currencyservice  log.error_count     25        0.0
16  recommendationservice  log.error_count     25        0.0
17         paymentservice  log.error_count     25        0.0
18        shippingservice  log.error_count     25        0.0
19                  redis  log.error_count      4        0.0, Empty DataFrame
Columns: [cmdb_id, error_count]
Index: [], Empty DataFrame
Columns: [timestamp, datetime_utc, cmdb_id, message_trunc]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
from pandas import Timestamp

# Reuse metric_df from previous state; if not present, load it
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['datetime_utc'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure datetime_utc exists and is timezone-aware UTC
if 'datetime_utc' not in metric_df.columns:
    metric_df['datetime_utc'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# 1) Compute global P95 and P5 and full_count for each (cmdb_id, kpi_name) pair among candidates (use full series)
metric_candidates_full = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# If empty, produce empty DataFrame with the required columns
if metric_candidates_full.empty:
    final_table = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','full_count','p95_value','p5_value',
        'window_count','anomalies_above_p95_count','anomalies_below_p5_count',
        'earliest_above_p95_timestamp','earliest_below_p5_timestamp',
        'max_value_in_window','min_value_in_window'
    ])
else:
    full_stats = metric_candidates_full.groupby(['cmdb_id','kpi_name'])['value'].agg(
        full_count='count',
        p95_value=lambda x: x.quantile(0.95),
        p5_value=lambda x: x.quantile(0.05)
    ).reset_index()

    # 2) Filter metric.csv to the incident window 2024-01-19 12:40:21 to 2024-01-19 13:10:21 UTC
    window_start = pd.to_datetime(1705668021, unit='s', utc=True)  # 2024-01-19 12:40:21 UTC
    window_end   = pd.to_datetime(1705668621, unit='s', utc=True)  # 2024-01-19 13:10:21 UTC

    metric_window = metric_candidates_full[
        (metric_candidates_full['datetime_utc'] >= window_start) &
        (metric_candidates_full['datetime_utc'] <= window_end)
    ].copy()

    # 3) For each pair compute window counts and anomaly counts and earliest timestamps and min/max in window
    if metric_window.empty:
        # Prepare window_stats with zeros/nulls for each pair
        window_stats = full_stats.copy()
        window_stats['window_count'] = 0
        window_stats['anomalies_above_p95_count'] = 0
        window_stats['anomalies_below_p5_count'] = 0
        window_stats['earliest_above_p95_timestamp'] = pd.NaT
        window_stats['earliest_below_p5_timestamp'] = pd.NaT
        window_stats['max_value_in_window'] = pd.NA
        window_stats['min_value_in_window'] = pd.NA
    else:
        # Merge thresholds into window rows for comparison
        window_with_thresholds = metric_window.merge(full_stats, on=['cmdb_id','kpi_name'], how='left')

        # Identify anomalies per row
        window_with_thresholds['above_p95'] = window_with_thresholds['value'] >= window_with_thresholds['p95_value']
        window_with_thresholds['below_p5'] = window_with_thresholds['value'] <= window_with_thresholds['p5_value']

        # Aggregate per pair
        agg_funcs = {
            'value': ['count', 'max', 'min'],
            'above_p95': 'sum',
            'below_p5': 'sum',
            'datetime_utc': lambda s: s.min()  # placeholder; will compute earliest timestamps separately
        }
        grouped = window_with_thresholds.groupby(['cmdb_id','kpi_name']).agg(
            window_count = ('value','count'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min'),
            anomalies_above_p95_count = ('above_p95','sum'),
            anomalies_below_p5_count = ('below_p5','sum')
        ).reset_index()

        # Earliest timestamps for above_p95 and below_p5
        def earliest_timestamp(df, cond_col):
            df_cond = df[df[cond_col]]
            if df_cond.empty:
                return pd.NaT
            return df_cond['datetime_utc'].min()

        earliest_records = window_with_thresholds.groupby(['cmdb_id','kpi_name']).apply(
            lambda g: pd.Series({
                'earliest_above_p95_timestamp': earliest_timestamp(g, 'above_p95'),
                'earliest_below_p5_timestamp': earliest_timestamp(g, 'below_p5')
            })
        ).reset_index()

        window_stats = grouped.merge(earliest_records, on=['cmdb_id','kpi_name'], how='left')

    # Combine full_stats and window_stats
    final = full_stats.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

    # Ensure columns exist and fill NaNs appropriately
    final_cols = [
        'cmdb_id','kpi_name','full_count','p95_value','p5_value',
        'window_count','anomalies_above_p95_count','anomalies_below_p5_count',
        'earliest_above_p95_timestamp','earliest_below_p5_timestamp',
        'max_value_in_window','min_value_in_window'
    ]
    for c in final_cols:
        if c not in final.columns:
            final[c] = pd.NA

    # Convert earliest timestamps to UTC strings (ISO) or keep NaT; keep as Timestamp objects for compact display
    # Fill numeric NaNs for counts with 0
    final['window_count'] = final['window_count'].fillna(0).astype(int)
    final['anomalies_above_p95_count'] = final['anomalies_above_p95_count'].fillna(0).astype(int)
    final['anomalies_below_p5_count'] = final['anomalies_below_p5_count'].fillna(0).astype(int)

    # Keep timestamps as UTC Timestamp or NaT
    # final already has Timestamp values from groupby min; ensure dtype
    final['earliest_above_p95_timestamp'] = pd.to_datetime(final['earliest_above_p95_timestamp'], utc=True)
    final['earliest_below_p5_timestamp'] = pd.to_datetime(final['earliest_below_p5_timestamp'], utc=True)

    # Sort by anomalies_above_p95_count desc then anomalies_below_p5_count desc, limit to top 20
    final_table = final[final_cols].sort_values(
        by=['anomalies_above_p95_count','anomalies_below_p5_count'], ascending=[False, False]
    ).reset_index(drop=True).head(20)

# Display the resulting table only (compact)
final_table

```
Out[2]:
```
summary = (
    "Summary (metric.csv) for candidate services:\n\n"
    "1) Broad observation:\n"
    "- Several socket and diskio KPIs across multiple services show constant values in the incident window\n"
    "  (checkoutservice.socket, currencyservice.diskio, emailservice.diskio, emailservice.socket,\n"
    "   productcatalogservice.diskio, productcatalogservice.socket, recommendationservice.diskio,\n"
    "   recommendationservice.socket). Each of these had window_count=8 and every window point equaled the global\n"
    "  p95 (and p5), with earliest occurrence at 2024-01-19 12:43:00 UTC. Max == Min for these series, indicating\n"
    "  a constant series (e.g., zeros or fixed counts) rather than transient spikes.\n\n"
    "2) Notable non-constant anomalies to investigate (higher priority):\n"
    "- currencyservice CPU: full p95 ~33.40, p5 ~29.66. In the incident window (8 points) there were 2 points >= p95\n"
    "  (earliest above at 2024-01-19 12:44:00 UTC) and 1 point <= p5 (earliest below at 2024-01-19 12:47:00 UTC).\n"
    "  Window max ~33.51, min ~29.61 — indicates CPU spikes during the window.\n"
    "- emailservice memory: full p95 ~43,508,950, p5 ~43,503,620. Window had 2 points >= p95 (earliest 12:43:00 UTC)\n"
    "  and 1 <= p5 (earliest 12:47:00 UTC). Max/min very close but show small upward excursions in memory.\n"
    "- currencyservice latency-50: p95 ~0.00418, p5 ~0.00385. Window had 2 points >= p95 (earliest 12:46:00 UTC),\n"
    "  suggesting brief latency increases (window max ~0.00426s).\n"
    "- productcatalogservice workload and recommendationservice workload: each had 2 points >= p95\n"
    "  (earliest at 2024-01-19 12:45:00 UTC) with elevated max values (productcatalog ~155, recommendation ~20),\n"
    "  indicating higher load spikes in the window.\n\n"
    "3) Additional single-point anomalies:\n"
    "- Several KPIs show a single point above p95 (e.g., currencyservice mem, currencyservice workload,\n"
    "  checkoutservice workload, emailservice workload, productcatalog cpu/mem). These have window_count=8 but only\n"
    "  1 point exceeding the global p95 — investigate if single-sample spikes correlate with traffic or deploys.\n\n"
    "4) Next steps:\n"
    "- Prioritize investigating currencyservice (CPU and latency) and emailservice memory spikes.\n"
    "- Review productcatalogservice and recommendationservice for workload spikes around 12:45 UTC.\n"
    "- Confirm that the socket/diskio series that equal p95/p5 are expected constants; if not, adjust thresholds or\n"
    "  instrumentation because p95==p5 makes anomaly detection ambiguous.\n\n"
    "Timestamps above are in UTC and earliest anomaly times come from the incident window 2024-01-19 12:40:21 to 2024-01-19 13:10:21 UTC."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  full_count     p95_value      p5_value  window_count  anomalies_above_p95_count  anomalies_below_p5_count earliest_above_p95_timestamp earliest_below_p5_timestamp  max_value_in_window  min_value_in_window
0         checkoutservice      socket          25  9.000000e+00  9.000000e+00             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00         9.000000e+00         9.000000e+00
1         currencyservice      diskio          25  0.000000e+00  0.000000e+00             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00         0.000000e+00         0.000000e+00
2            emailservice      diskio          25  0.000000e+00  0.000000e+00             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00         0.000000e+00         0.000000e+00
3            emailservice      socket          25  3.000000e+00  3.000000e+00             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00         3.000000e+00         3.000000e+00
4   productcatalogservice      diskio          25  0.000000e+00  0.000000e+00             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00         0.000000e+00         0.000000e+00
5   productcatalogservice      socket          25  3.000000e+00  3.000000e+00             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00         3.000000e+00         3.000000e+00
6   recommendationservice      diskio          25  0.000000e+00  0.000000e+00             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00         0.000000e+00         0.000000e+00
7   recommendationservice      socket          25  4.000000e+00  4.000000e+00             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00         4.000000e+00         4.000000e+00
8         currencyservice         cpu          25  3.339752e+01  2.966166e+01             8                          2                         1    2024-01-19 12:44:00+00:00   2024-01-19 12:47:00+00:00         3.350784e+01         2.960954e+01
9            emailservice         mem          25  4.350895e+07  4.350362e+07             8                          2                         1    2024-01-19 12:43:00+00:00   2024-01-19 12:47:00+00:00         4.350929e+07         4.350362e+07
10        currencyservice  latency-50          25  4.183259e-03  3.848286e-03             8                          2                         0    2024-01-19 12:46:00+00:00                         NaT         4.257619e-03         3.903377e-03
11  productcatalogservice    workload          25  1.518811e+02  1.347169e+02             8                          2                         0    2024-01-19 12:45:00+00:00                         NaT         1.551990e+02         1.363335e+02
12  recommendationservice    workload          25  1.983565e+01  1.755813e+01             8                          2                         0    2024-01-19 12:45:00+00:00                         NaT         2.008538e+01         1.781892e+01
13        currencyservice      socket          25  3.133333e+00  3.000000e+00             8                          1                         7    2024-01-19 12:50:00+00:00   2024-01-19 12:43:00+00:00         3.166667e+00         3.000000e+00
14        currencyservice         mem          25  4.555095e+07  4.462510e+07             8                          1                         1    2024-01-19 12:49:00+00:00   2024-01-19 12:44:00+00:00         4.556595e+07         4.446242e+07
15        currencyservice    workload          25  8.205031e+01  7.496790e+01             8                          1                         1    2024-01-19 12:47:00+00:00   2024-01-19 12:48:00+00:00         8.206207e+01         7.491570e+01
16        checkoutservice    workload          25  1.120730e+00  6.320000e-01             8                          1                         0    2024-01-19 12:45:00+00:00                         NaT         1.301200e+00         6.599167e-01
17           emailservice    workload          25  1.121450e+00  6.500233e-01             8                          1                         0    2024-01-19 12:45:00+00:00                         NaT         1.266600e+00         6.832500e-01
18  productcatalogservice         cpu          25  3.869884e+00  3.157192e+00             8                          1                         0    2024-01-19 12:49:00+00:00                         NaT         3.893195e+00         3.300981e+00
19  productcatalogservice         mem          25  1.066215e+07  1.024558e+07             8                          1                         0    2024-01-19 12:43:00+00:00                         NaT         1.087057e+07         1.028581e+07```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse trace_df if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['datetime_utc'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure datetime_utc exists and is timezone-aware UTC
if 'datetime_utc' not in trace_df.columns:
    trace_df['datetime_utc'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Filter full series to candidates
trace_candidates_full = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()

# If empty, prepare empty DataFrame with required columns
cols = [
    'cmdb_id','trace_name','full_count','p95_value','p5_value',
    'window_count','anomalies_above_p95_count','anomalies_below_p5_count',
    'earliest_above_p95_timestamp','earliest_below_p5_timestamp',
    'max_value_in_window','min_value_in_window'
]

if trace_candidates_full.empty:
    final_table = pd.DataFrame(columns=cols)
else:
    # 1) Compute global P95 and P5 and full_count per (cmdb_id, trace_name)
    full_stats = trace_candidates_full.groupby(['cmdb_id','trace_name'])['value'].agg(
        full_count='count',
        p95_value=lambda x: x.quantile(0.95),
        p5_value=lambda x: x.quantile(0.05)
    ).reset_index()

    # 2) Filter to incident window
    window_start = pd.to_datetime(1705668021, unit='s', utc=True)  # 2024-01-19 12:40:21 UTC
    window_end   = pd.to_datetime(1705668621, unit='s', utc=True)  # 2024-01-19 13:10:21 UTC

    trace_window = trace_candidates_full[
        (trace_candidates_full['datetime_utc'] >= window_start) &
        (trace_candidates_full['datetime_utc'] <= window_end)
    ].copy()

    # 3) Compute window stats and anomaly counts
    if trace_window.empty:
        # fill zeros/nulls for each pair
        window_stats = full_stats.copy()
        window_stats['window_count'] = 0
        window_stats['anomalies_above_p95_count'] = 0
        window_stats['anomalies_below_p5_count'] = 0
        window_stats['earliest_above_p95_timestamp'] = pd.NaT
        window_stats['earliest_below_p5_timestamp'] = pd.NaT
        window_stats['max_value_in_window'] = pd.NA
        window_stats['min_value_in_window'] = pd.NA
    else:
        # Merge thresholds into window rows
        window_with_thresholds = trace_window.merge(full_stats, on=['cmdb_id','trace_name'], how='left')

        # Mark anomalies
        window_with_thresholds['above_p95'] = window_with_thresholds['value'] >= window_with_thresholds['p95_value']
        window_with_thresholds['below_p5'] = window_with_thresholds['value'] <= window_with_thresholds['p5_value']

        # Aggregate counts and min/max
        grouped = window_with_thresholds.groupby(['cmdb_id','trace_name']).agg(
            window_count = ('value','count'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min'),
            anomalies_above_p95_count = ('above_p95','sum'),
            anomalies_below_p5_count = ('below_p5','sum')
        ).reset_index()

        # Earliest timestamps for anomalies
        def earliest_ts(df, cond_col):
            sub = df[df[cond_col]]
            if sub.empty:
                return pd.NaT
            return sub['datetime_utc'].min()

        earliest = window_with_thresholds.groupby(['cmdb_id','trace_name']).apply(
            lambda g: pd.Series({
                'earliest_above_p95_timestamp': earliest_ts(g, 'above_p95'),
                'earliest_below_p5_timestamp': earliest_ts(g, 'below_p5')
            })
        ).reset_index()

        window_stats = grouped.merge(earliest, on=['cmdb_id','trace_name'], how='left')

    # Combine full_stats and window_stats
    final = full_stats.merge(window_stats, on=['cmdb_id','trace_name'], how='left')

    # Ensure all required columns exist
    for c in cols:
        if c not in final.columns:
            final[c] = pd.NA

    # Fill numeric NaNs for counts with 0 and convert types
    final['window_count'] = final['window_count'].fillna(0).astype(int)
    final['anomalies_above_p95_count'] = final['anomalies_above_p95_count'].fillna(0).astype(int)
    final['anomalies_below_p5_count'] = final['anomalies_below_p5_count'].fillna(0).astype(int)

    # Ensure earliest timestamps are proper UTC Timestamps or NaT
    final['earliest_above_p95_timestamp'] = pd.to_datetime(final['earliest_above_p95_timestamp'], utc=True)
    final['earliest_below_p5_timestamp'] = pd.to_datetime(final['earliest_below_p5_timestamp'], utc=True)

    # Reorder/select columns and sort as requested, limit to top 20
    final_table = final[cols].sort_values(
        by=['anomalies_above_p95_count','anomalies_below_p5_count'], ascending=[False, False]
    ).reset_index(drop=True).head(20)

# Display only the resulting table
final_table

```
Out[3]:
```
summary = (
    "Trace summary for candidate services (UTC):\n\n"
    "- The top trace results for checkoutservice, currencyservice, emailservice, productcatalogservice, and recommendationservice\n"
    "  are all 'error_rate' traces.\n"
    "- For every returned (cmdb_id, trace_name): global P95 = 0.0 and P5 = 0.0. In the incident window (2024-01-19 12:40:21 to 13:10:21 UTC)\n"
    "  each trace had window_count = 8 and every window value = 0.0. As a result anomalies_above_p95_count = 8 and\n"
    "  anomalies_below_p5_count = 8 (because p95 == p5 == 0 and values == 0). The earliest matching timestamp shown is\n"
    "  2024-01-19 12:43:00+00:00, and max/min in-window values are 0.0.\n\n"
    "Interpretation and next steps:\n"
    "- These traces show constant zero error rates (no observed errors) in both the full series and the incident window.\n"
    "- Because p95 == p5 == value == 0, this metric cannot indicate an increase in errors—either there truly were no errors\n"
    "  or error_rate instrumentation/aggregation yields constant zeros. This makes anomaly detection on error_rate ambiguous.\n"
    "- To find problems, inspect non-error trace metrics (durations, row_count, latency traces) and the earlier metric results\n"
    "  (which showed some elevated checkoutservice durations). Also verify trace instrumentation and sampling to ensure error\n"
    "  rates would be reported when present.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  full_count  p95_value  p5_value  window_count  anomalies_above_p95_count  anomalies_below_p5_count earliest_above_p95_timestamp earliest_below_p5_timestamp  max_value_in_window  min_value_in_window
0         checkoutservice        trace.from_checkoutservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
1         checkoutservice        trace.from_frontendservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
2         checkoutservice                   trace.from_root.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
3         checkoutservice          trace.to_checkoutservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
4         checkoutservice          trace.to_currencyservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
5         checkoutservice             trace.to_emailservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
6         checkoutservice           trace.to_paymentservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
7         checkoutservice    trace.to_productcatalogservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
8         currencyservice        trace.from_checkoutservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
9         currencyservice        trace.from_frontendservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
10        currencyservice                   trace.from_root.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
11           emailservice        trace.from_checkoutservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
12           emailservice                   trace.from_root.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
13  productcatalogservice        trace.from_checkoutservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
14  productcatalogservice        trace.from_frontendservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
15  productcatalogservice  trace.from_recommendationservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
16  productcatalogservice                   trace.from_root.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
17  recommendationservice        trace.from_frontendservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
18  recommendationservice  trace.from_recommendationservice.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0
19  recommendationservice                   trace.from_root.error_rate          25        0.0       0.0             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00                  0.0                  0.0```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse trace_df from previous state if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['datetime_utc'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure datetime column exists and is UTC-aware
if 'datetime_utc' not in trace_df.columns:
    trace_df['datetime_utc'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Focus services
services = ['checkoutservice', 'currencyservice']

# 1) Compute global P95 and P5 and full_count per (cmdb_id, trace_name) for these services
trace_focus_full = trace_df[trace_df['cmdb_id'].isin(services)].copy()

cols = [
    'cmdb_id','trace_name','full_count','p95_value','p5_value',
    'window_count','anomalies_above_p95_count','anomalies_below_p5_count',
    'earliest_above_p95_timestamp','earliest_below_p5_timestamp',
    'max_value_in_window','min_value_in_window'
]

if trace_focus_full.empty:
    final_table = pd.DataFrame(columns=cols)
else:
    full_stats = trace_focus_full.groupby(['cmdb_id','trace_name'])['value'].agg(
        full_count='count',
        p95_value=lambda x: x.quantile(0.95),
        p5_value=lambda x: x.quantile(0.05)
    ).reset_index()

    # 2) Filter to incident window (UTC)
    window_start = pd.to_datetime(1705668021, unit='s', utc=True)  # 2024-01-19 12:40:21 UTC
    window_end   = pd.to_datetime(1705668621, unit='s', utc=True)  # 2024-01-19 13:10:21 UTC

    trace_window = trace_focus_full[
        (trace_focus_full['datetime_utc'] >= window_start) &
        (trace_focus_full['datetime_utc'] <= window_end)
    ].copy()

    # 3) Compute window stats and anomaly counts
    if trace_window.empty:
        window_stats = full_stats.copy()
        window_stats['window_count'] = 0
        window_stats['anomalies_above_p95_count'] = 0
        window_stats['anomalies_below_p5_count'] = 0
        window_stats['earliest_above_p95_timestamp'] = pd.NaT
        window_stats['earliest_below_p5_timestamp'] = pd.NaT
        window_stats['max_value_in_window'] = pd.NA
        window_stats['min_value_in_window'] = pd.NA
    else:
        # Merge thresholds into window rows
        window_with_thresholds = trace_window.merge(full_stats, on=['cmdb_id','trace_name'], how='left')

        # Mark anomalies
        window_with_thresholds['above_p95'] = window_with_thresholds['value'] >= window_with_thresholds['p95_value']
        window_with_thresholds['below_p5'] = window_with_thresholds['value'] <= window_with_thresholds['p5_value']

        # Aggregate counts and min/max
        grouped = window_with_thresholds.groupby(['cmdb_id','trace_name']).agg(
            window_count = ('value','count'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min'),
            anomalies_above_p95_count = ('above_p95','sum'),
            anomalies_below_p5_count = ('below_p5','sum')
        ).reset_index()

        # Earliest timestamps for anomalies
        def earliest_ts(subdf, cond_col):
            s = subdf[subdf[cond_col]]
            if s.empty:
                return pd.NaT
            return s['datetime_utc'].min()

        earliest = window_with_thresholds.groupby(['cmdb_id','trace_name']).apply(
            lambda g: pd.Series({
                'earliest_above_p95_timestamp': earliest_ts(g, 'above_p95'),
                'earliest_below_p5_timestamp': earliest_ts(g, 'below_p5')
            })
        ).reset_index()

        window_stats = grouped.merge(earliest, on=['cmdb_id','trace_name'], how='left')

    # Combine full and window stats
    final = full_stats.merge(window_stats, on=['cmdb_id','trace_name'], how='left')

    # Ensure all required columns exist
    for c in cols:
        if c not in final.columns:
            final[c] = pd.NA

    # Fill numeric NaNs for counts with 0 and cast
    final['window_count'] = final['window_count'].fillna(0).astype(int)
    final['anomalies_above_p95_count'] = final['anomalies_above_p95_count'].fillna(0).astype(int)
    final['anomalies_below_p5_count'] = final['anomalies_below_p5_count'].fillna(0).astype(int)

    # Ensure earliest timestamps are proper UTC Timestamps or NaT
    final['earliest_above_p95_timestamp'] = pd.to_datetime(final['earliest_above_p95_timestamp'], utc=True)
    final['earliest_below_p5_timestamp'] = pd.to_datetime(final['earliest_below_p5_timestamp'], utc=True)

    # Select and order columns, sort and limit to top 20
    final_table = final[cols].sort_values(
        by=['anomalies_above_p95_count','anomalies_below_p5_count'], ascending=[False, False]
    ).reset_index(drop=True).head(20)

# Display the resulting compact table
final_table

```
Out[4]:
```
Summary (checkoutservice and currencyservice traces, UTC):

- Error rates: All error_rate traces are constant zero (global P95 = P5 = 0.0 and window values = 0.0). They do not indicate failures.

- Highest-impact anomaly — currencyservice row_count:
  - trace: currencyservice / trace.from_root.row_count
  - Full P95 = 12, P5 ≈ 8.8. In the incident window there were 8 points and 7 of them >= P95 (earliest above: 2024-01-19 12:44:00 UTC). Window max = 12, min = 8.
  - Interpretation: sustained elevated request/row counts hitting currencyservice during the window (traffic spike).

- Latency/duration anomalies between checkout and currency:
  - trace.to_currencyservice.duration_mean (checkoutservice) and trace.from_checkoutservice.duration_mean (currencyservice):
    - Full P95 ≈ 0.001224 s. In the window each had 2 points ≥ P95 (earliest above: 2024-01-19 12:48:00 UTC) and 2 points ≤ P5 (earliest below: 12:44:00 UTC). Window max ≈ 0.00278 s, min ≈ 0.000145 s.
  - trace.*.duration_p95 variants for checkout <-> currency also show 2 points above P95 with earliest above around 12:46:00 UTC; one series (checkout -> currency duration_p95) had a window max = 0.015111 s.
  - Interpretation: intermittent latency increases on requests between checkoutservice and currencyservice around ~12:46–12:48 UTC.

- Other notable points:
  - Some traces (to_emailservice) show 1 above / 1 below P5 in-window — minor single-sample excursions.
  - Overall, error_rate traces are uninformative (all zeros); the meaningful anomalies are elevated traffic to currencyservice and corresponding increased durations on calls between checkoutservice and currencyservice.

Recommended next steps:
- Investigate currencyservice load (incoming traffic, autoscaling, resource usage) starting ~12:44 UTC.
- Inspect checkout ↔ currency RPCs around 12:46–12:48 UTC for increased latency and any downstream/resource contention.
- Check application logs and host metrics (CPU, memory, network) for currencyservice and checkoutservice to correlate with the trace spikes.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                 trace_name  full_count  p95_value  p5_value  window_count  anomalies_above_p95_count  anomalies_below_p5_count earliest_above_p95_timestamp earliest_below_p5_timestamp  max_value_in_window  min_value_in_window
0   checkoutservice      trace.from_checkoutservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
1   checkoutservice      trace.from_frontendservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
2   checkoutservice                 trace.from_root.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
3   checkoutservice        trace.to_checkoutservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
4   checkoutservice        trace.to_currencyservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
5   checkoutservice           trace.to_emailservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
6   checkoutservice         trace.to_paymentservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
7   checkoutservice  trace.to_productcatalogservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
8   currencyservice      trace.from_checkoutservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
9   currencyservice      trace.from_frontendservice.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
10  currencyservice                 trace.from_root.error_rate          25   0.000000  0.000000             8                          8                         8    2024-01-19 12:43:00+00:00   2024-01-19 12:43:00+00:00             0.000000             0.000000
11  currencyservice                  trace.from_root.row_count          25  12.000000  8.800000             8                          7                         1    2024-01-19 12:44:00+00:00   2024-01-19 12:43:00+00:00            12.000000             8.000000
12  checkoutservice     trace.to_currencyservice.duration_mean          25   0.001224  0.000146             8                          2                         2    2024-01-19 12:48:00+00:00   2024-01-19 12:44:00+00:00             0.002777             0.000145
13  currencyservice   trace.from_checkoutservice.duration_mean          25   0.001224  0.000146             8                          2                         2    2024-01-19 12:48:00+00:00   2024-01-19 12:44:00+00:00             0.002777             0.000145
14  currencyservice               trace.from_root.duration_p95          25   0.000136  0.000059             8                          2                         1    2024-01-19 12:46:00+00:00   2024-01-19 12:43:00+00:00             0.000164             0.000059
15  checkoutservice      trace.to_currencyservice.duration_p95          25   0.000373  0.000210             8                          2                         0    2024-01-19 12:46:00+00:00                         NaT             0.015111             0.000210
16  currencyservice    trace.from_checkoutservice.duration_p95          25   0.000373  0.000210             8                          2                         0    2024-01-19 12:46:00+00:00                         NaT             0.015111             0.000210
17  currencyservice              trace.from_root.duration_mean          25   0.000069  0.000051             8                          2                         0    2024-01-19 12:46:00+00:00                         NaT             0.000070             0.000051
18  checkoutservice        trace.to_emailservice.duration_mean          25   0.000326  0.000264             8                          1                         1    2024-01-19 12:47:00+00:00   2024-01-19 12:44:00+00:00             0.000333             0.000263
19  checkoutservice         trace.to_emailservice.duration_p95          25   0.000592  0.000364             8                          1                         1    2024-01-19 12:47:00+00:00   2024-01-19 12:44:00+00:00             0.000606             0.000362```
```